# Databricks/PySpark dla Data Analyst/Engineer — Moduł 8: UDF i funkcje niestandardowe

Czasami żadna funkcja z `pyspark.sql.functions` nie robi dokładnie tego, czego
potrzebujemy -- wtedy sięgamy po **UDF** (*User Defined Function*, funkcję zdefiniowaną
przez użytkownika w zwykłym Pythonie). Ten moduł pokazuje, jak je pisać, ale przede
wszystkim -- dlaczego warto ich UNIKAĆ, kiedy się tylko da.

## Spis treści
1. [Kiedy naprawdę potrzebujesz UDF (a kiedy nie)](#sec1)
2. [Zwykły Python UDF](#sec2)
3. [Koszt wydajnościowy UDF-ów](#sec3)
4. [Pandas UDF -- wektoryzowana alternatywa](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Kiedy naprawdę potrzebujesz UDF (a kiedy nie)

Zanim napiszesz UDF, sprawdź, czy da się to samo osiągnąć funkcjami wbudowanymi
(`pyspark.sql.functions`, Moduł 3) -- zestaw `when/otherwise`, funkcje tekstowe, daty,
matematyczne i tak dalej pokrywa zaskakująco dużo przypadków. UDF ma sens dopiero, gdy
logika jest naprawdę niestandardowa: skomplikowany algorytm, wywołanie zewnętrznej
biblioteki Pythona (np. `re` do złożonych wyrażeń regularnych, biblioteka NLP), albo
logika trudna do wyrażenia jako pojedyncze wyrażenie SQL.

> ⚠️ **Najczęstszy błąd początkujących: UDF tam, gdzie starczyłoby when/otherwise**
>
> Przykład: `@udf` klasyfikujący cenę na `'Tani'`/`'Sredni'`/`'Drogi'` przez `if/elif` w Pythonie -- to DOKŁADNIE to, co `F.when().when().otherwise()` z Modułu 3 robi szybciej i bez żadnego UDF-a. Zawsze najpierw sprawdź funkcje wbudowane.

<a id="sec2"></a>
## 2. Zwykły Python UDF

`@F.udf(typ_zwracany)` zamienia zwykłą funkcję Pythona w UDF-a, którego można użyć w
`withColumn`/`select`. WAŻNE: musisz jawnie podać typ zwracany (`pyspark.sql.types`) --
Spark nie potrafi go sam odgadnąć z kodu Pythona.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

@F.udf(StringType())
def skroc_nazwe_miasta(miasto):
    if miasto is None:
        return None
    return miasto[:3].upper()

klienci.withColumn("kod_miasta", skroc_nazwe_miasta(F.col("miasto"))).select(
    "klient_id", "miasto", "kod_miasta"
).show(5)


> ⚡ **Rejestracja UDF-a do użycia w Spark SQL**
>
> Jeśli chcesz użyć tej samej funkcji w zapytaniu `spark.sql(...)` (Moduł 5), zarejestruj ją przez `spark.udf.register("nazwa_sql", skroc_nazwe_miasta)` -- dopiero wtedy jest widoczna jako zwykła funkcja SQL w tekstowych zapytaniach.

<a id="sec3"></a>
## 3. Koszt wydajnościowy UDF-ów

Funkcje wbudowane (`pyspark.sql.functions`) działają WEWNĄTRZ JVM -- Catalyst zna ich
logikę i może je zoptymalizować razem z resztą planu. Zwykły Python UDF wymaga
SERIALIZACJI każdego wiersza z JVM do procesu Pythona, wykonania funkcji w Pythonie, i
DESERIALIZACJI wyniku z powrotem do JVM -- dla każdego wiersza osobno. To poważny koszt
na dużych zbiorach danych.

In [ ]:
# explain() UDF-a nie pokazuje, CO robi funkcja -- Catalyst widzi ja jako "czarna skrzynke"
klienci.withColumn("kod_miasta", skroc_nazwe_miasta(F.col("miasto"))).explain()


> ⚠️ **PythonUDF w planie wykonania -- sygnał ostrzegawczy**
>
> Jeśli w `.explain()` widzisz węzeł `BatchEvalPython` albo podobny -- to znak, że Spark musi przełączać się między JVM a Pythonem dla tej kolumny. Pojedynczy UDF na małych danych to nic groźnego, ale wiele UDF-ów w łańcuchu na dużym zbiorze danych bywa głównym wąskim gardłem całego potoku -- pierwsze miejsce do sprawdzenia, gdy coś działa wolniej niż się spodziewasz.

<a id="sec4"></a>
## 4. Pandas UDF -- wektoryzowana alternatywa

**Pandas UDF** (dawniej *Vectorized UDF*) przetwarza dane w PORCJACH (jako `pandas.Series`),
korzystając z **Apache Arrow** do dużo wydajniejszego transferu danych między JVM a
Pythonem -- zamiast wiersz po wierszu. Składnia jest podobna, ale funkcja operuje na
całej kolumnie naraz.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def skroc_nazwe_miasta_pandas(miasto: pd.Series) -> pd.Series:
    return miasto.str[:3].str.upper()

# pandas UDF korzysta z Apache Arrow do transferu danych JVM <-> Python -- to wymaga
# zgodnej wersji Javy i biblioteki Arrow dolaczonej do Sparka (patrz uwaga ponizej).
# Na niekompatybilnym zestawieniu Java/Arrow rzuci wyjatek -- w takim wypadku ponizszy
# blok try/except grzecznie to zglosi, zamiast wywalac cale wykonanie notatnika.
try:
    klienci.withColumn(
        "kod_miasta_pandas", skroc_nazwe_miasta_pandas(F.col("miasto"))
    ).select("klient_id", "miasto", "kod_miasta_pandas").show(5)
except Exception as e:
    print("Pandas UDF nie dziala w tym srodowisku (niezgodnosc Java/Apache Arrow).")
    print(f"Szczegoly: {type(e).__name__}")


> ⚠️ **Pandas UDF wymaga zgodnej wersji Javy i Apache Arrow**
>
> Pandas UDF-y opierają się na **Apache Arrow** do przekazywania danych między JVM a Pythonem -- to wymaga, żeby wersja Javy, na której działa Spark, była zgodna z wersją biblioteki Arrow dołączonej do danej wersji PySpark. Na bardzo nowych wersjach Javy (np. JDK 21) uruchomionych z Arrow w starszej wersji (jak w tym kursie) można dostać błąd typu `UnsupportedOperationException` przy próbie wywołania pandas UDF-a -- to ograniczenie środowiska, nie błąd w kodzie. W Databricks Java i Arrow są zawsze dobrane tak, żeby ze sobą współpracowały, więc tam ten sam kod zadziała bez żadnych poprawek.

> 🧱 **Kiedy pandas UDF naprawdę się opłaca**
>
> Różnica wydajności jest znikoma dla małych zbiorów danych (jak nasz w tym kursie), ale rośnie znacząco na dużych, rozproszonych danych -- zwłaszcza gdy logika w Pythonie korzysta z bibliotek zoptymalizowanych wektorowo (`numpy`, `pandas`, modele ML). W Databricks pandas UDF-y są standardowym narzędziem do integrowania niestandardowej logiki Pythona/ML w potokach Spark bez rezygnowania z wydajności.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- kiedy UDF jest naprawdę potrzebny -- i że najpierw zawsze warto sprawdzić funkcje wbudowane,
- zwykły Python UDF (`@F.udf(typ_zwracany)`) -- składnia i jawny typ zwracany,
- koszt wydajnościowy: serializacja/deserializacja wiersz po wierszu między JVM a Pythonem,
- pandas UDF (`@pandas_udf`) -- wektoryzowana alternatywa oparta o Apache Arrow.

### 📝 Ćwiczenie 1: UDF liczący długość regionu

Napisz zwykły Python UDF zwracający długość napisu (`len()`) kolumny `region` z `pracownicy` -- pomyśl, czy dałoby się to zrobić bez UDF-a (podpowiedź: tak, `F.length()`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql.types import IntegerType

@F.udf(IntegerType())
def dlugosc_regionu(region):
    return len(region) if region is not None else None

pracownicy.withColumn("dlugosc", dlugosc_regionu(F.col("region"))).show()
# Bez UDF-a: pracownicy.withColumn("dlugosc", F.length("region")).show()
```

</details>

### 📝 Ćwiczenie 2: UDF z niestandardową logiką

Napisz Python UDF, który dla kolumny `status` zwraca emoji: `'✅'` dla `'Dostarczone'`, `'📦'` dla `'Wyslane'`, `'❌'` dla `'Anulowane'`, `'🕐'` w pozostałych przypadkach.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
@F.udf(StringType())
def status_emoji(status):
    mapa = {"Dostarczone": "OK", "Wyslane": "W DRODZE", "Anulowane": "ANULOWANE"}
    return mapa.get(status, "OCZEKUJE")

zamowienia.withColumn("status_opis", status_emoji(F.col("status"))).select(
    "zamowienie_id", "status", "status_opis"
).show(10)
```

</details>

### 📝 Ćwiczenie 3: Ten sam UDF jako pandas UDF

Przepisz UDF z ćwiczenia 2 jako pandas UDF, operujący na całej kolumnie (`pandas.Series`) naraz zamiast wiersz po wierszu.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
@pandas_udf(StringType())
def status_opis_pandas(status: pd.Series) -> pd.Series:
    mapa = {"Dostarczone": "OK", "Wyslane": "W DRODZE", "Anulowane": "ANULOWANE"}
    return status.map(mapa).fillna("OCZEKUJE")

zamowienia.withColumn("status_opis", status_opis_pandas(F.col("status"))).select(
    "zamowienie_id", "status", "status_opis"
).show(10)
```

</details>

### 📝 Ćwiczenie 4: Rejestracja UDF-a do Spark SQL

Zarejestruj UDF `dlugosc_regionu` z ćwiczenia 1 pod nazwą SQL `dlugosc_sql` i użyj go w zapytaniu `spark.sql(...)` na widoku `pracownicy_view` (najpierw zarejestruj widok przez `createOrReplaceTempView`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
pracownicy.createOrReplaceTempView("pracownicy_view")
spark.udf.register("dlugosc_sql", dlugosc_regionu)

spark.sql("""
    SELECT pracownik_id, region, dlugosc_sql(region) AS dlugosc
    FROM pracownicy_view
""").show()
```

</details>

> 🔥 **Wyzwanie: porównanie planu wykonania -- zwykły UDF vs. funkcje wbudowane**
>
> Skategoryzuj `produkty` na `przedzial_cenowy` DWOMA sposobami: (1) zwykłym Python UDF-em z `if/elif`, (2) funkcjami wbudowanymi `when/otherwise` (jak w Module 3). Wywołaj `.explain()` na obu wynikach i porównaj -- w którym planie widać `BatchEvalPython` (albo podobny węzeł), a w którym nie?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
@F.udf(StringType())
def przedzial_cenowy_udf(cena):
    if cena < 100:
        return "Tani"
    elif cena <= 500:
        return "Sredni"
    return "Drogi"

wersja_udf = produkty.withColumn("przedzial_cenowy", przedzial_cenowy_udf(F.col("cena")))
wersja_wbudowana = produkty.withColumn(
    "przedzial_cenowy",
    F.when(F.col("cena") < 100, "Tani")
    .when(F.col("cena") <= 500, "Sredni")
    .otherwise("Drogi"),
)

print("=== Wersja z UDF-em ===")
wersja_udf.explain()
print("=== Wersja z funkcjami wbudowanymi ===")
wersja_wbudowana.explain()
```

W planie wersji z UDF-em powinien pojawic sie wezel zwiazany z ewaluacja Pythona (np. BatchEvalPython) -- w wersji z when/otherwise go nie ma, bo cala logika zostaje wewnatrz JVM i jest w pelni zoptymalizowana przez Catalyst.

</details>

## Co dalej?

W **Module 9** zajmiemy się optymalizacją i wydajnością: partycjonowanie danych w
pamięci, `cache()`/`persist()`, przegląd Catalyst/Tungsten i dokładniejsze czytanie
planów z `.explain()`.